# Lab 9B: Convolutional Autoencoders — Latent Space, Denoising and Skip Connections

**Before you start: Runtime → Change runtime type → GPU.**

This is the second of four notebooks (see 9A for the overview). In 9A, the autoencoder treated images
as vectors. Here we use **convolutions**, look closely at the **latent space**, and use autoencoders
for something practical: **denoising**.

| Part | You will… |
|---|---|
| 1. A convolutional autoencoder | get the shapes of transposed convolutions right, and explore a 2D latent space: scatter plot, latent map and interpolation |
| 2. A denoising autoencoder | train an autoencoder to remove noise, and find out where it breaks down |
| 3. Skip connections (U-Net) | add skip connections, and find out what they do to fine details, and to the bottleneck |
| 4. Exam-style questions | check your understanding |

**Reminder:** we use 10% of MNIST and small networks, so that each model trains in about a minute.

## Setup

In [ ]:
import copy
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---- Data: 10% of the MNIST training set, and 2,000 test images (used as a validation set) ----
FRACTION = 0.1
mnist_train = torchvision.datasets.MNIST('./data', train=True, download=True)
mnist_test = torchvision.datasets.MNIST('./data', train=False, download=True)
idx = torch.randperm(len(mnist_train), generator=torch.Generator().manual_seed(0))[:int(FRACTION * len(mnist_train))]

# The whole (small) dataset fits on the GPU, so we skip the DataLoader and index tensors directly.
X_train = (mnist_train.data[idx].float() / 255).unsqueeze(1).to(device)    # (6000, 1, 28, 28), values in [0, 1]
y_train = mnist_train.targets[idx].to(device)                               # labels: used ONLY for colouring plots
X_test = (mnist_test.data[:2000].float() / 255).unsqueeze(1).to(device)    # (2000, 1, 28, 28)
y_test = mnist_test.targets[:2000].to(device)
print('X_train:', tuple(X_train.shape), ' X_test:', tuple(X_test.shape))


def batches(n, batch_size=64, shuffle=True):
    # Mini-batches of indices: the same job a DataLoader(shuffle=True) does for you.
    order = torch.randperm(n, device=device) if shuffle else torch.arange(n, device=device)
    return order.split(batch_size)


def recon_loss(x_hat, x):
    # J = (1/n) * sum_i ||x_hat_i - x_i||^2 : squared error summed over pixels, averaged over the batch
    return ((x_hat - x) ** 2).flatten(1).sum(1).mean()


def show_images(x, nrow=10, title=None, ax=None, figsize=None):
    # Show a batch of (N, 1, 28, 28) images as one grid.
    x = x.detach().float().cpu().clamp(0, 1)
    grid = torchvision.utils.make_grid(x, nrow=nrow, padding=1, pad_value=1)[0]
    if ax is None:
        plt.figure(figsize=figsize or (nrow * 0.6, (len(x) + nrow - 1) // nrow * 0.6))
        ax = plt.gca()
    ax.imshow(grid, cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    if title:
        ax.set_title(title)


def plot_latent(z, labels, ax=None, title=None, lim=None):
    # Scatter plot of 2D latent codes, coloured by the true digit label (labels are never used in training).
    z, labels = z.detach().cpu().numpy(), labels.cpu().numpy()
    ax = ax or plt.gca()
    sc = ax.scatter(z[:, 0], z[:, 1], c=labels, cmap='tab10', s=4, vmin=-0.5, vmax=9.5)
    plt.colorbar(sc, ax=ax, ticks=range(10))
    if lim is not None:
        ax.set_xlim(lim[0]); ax.set_ylim(lim[1])
    ax.set_xlabel('z[0]'); ax.set_ylabel('z[1]')
    if title:
        ax.set_title(title)


@torch.no_grad()
def plot_latent_grid(decode, xlim, ylim, n=20, ax=None, title=None):
    # Decode an n x n grid of latent vectors and show each decoded image at its position in latent space.
    gx = torch.linspace(xlim[0], xlim[1], n)
    gy = torch.linspace(ylim[1], ylim[0], n)            # top row = largest z[1], as in a scatter plot
    z = torch.stack(torch.meshgrid(gy, gx, indexing='ij')[::-1], -1).view(-1, 2).to(device)
    imgs = decode(z).view(n, n, 28, 28).cpu()
    canvas = imgs.permute(0, 2, 1, 3).reshape(n * 28, n * 28)
    ax = ax or plt.gca()
    ax.imshow(canvas, cmap='gray', extent=[xlim[0], xlim[1], ylim[0], ylim[1]], aspect='auto', vmin=0, vmax=1)
    ax.set_xlabel('z[0]'); ax.set_ylabel('z[1]')
    if title:
        ax.set_title(title)

In [ ]:
def train_model(model, epochs=20, lr=1e-3, noise_std=None, verbose=True):
    # Trains model(x) -> x_hat to reconstruct the CLEAN image. If noise_std is set, the INPUT is corrupted first.
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for b in batches(len(X_train), 64):
            x = X_train[b]
            x_in = x if noise_std is None else add_noise(x, noise_std)
            loss = recon_loss(model(x_in), x)
            opt.zero_grad(); loss.backward(); opt.step()
        if verbose and (epoch + 1) % 10 == 0:
            print(f'epoch {epoch + 1:3d}   test loss {evaluate(model, noise_std):7.2f}')
    model.eval()
    return model


@torch.no_grad()
def evaluate(model, noise_std=None, seed=123):
    # Average reconstruction loss on the test set (against the clean images), with fixed noise.
    model.eval()
    g = torch.Generator(device=device).manual_seed(seed)
    x_in = X_test if not noise_std else (X_test + noise_std * torch.randn(X_test.shape, generator=g, device=device)).clamp(0, 1)
    return recon_loss(model(x_in), X_test).item()


def add_noise(x, std):          # replaced in Part 2 (Task 2a)
    raise NotImplementedError('You implement this in Task 2a')

---
# Part 1 — A convolutional autoencoder

The model below is the convolutional autoencoder from the lecture notes (Code 1). The encoder
downsamples with **strided convolutions**; the decoder upsamples with **transposed convolutions**
(`nn.ConvTranspose2d`).

### 🔗 Theory ↔ code: transposed convolutions

A transposed convolution is the learnable alternative to simple upsampling (Lecture 7). One way to
picture it: every input pixel "stamps" a copy of a learnable $k \times k$ filter, scaled by the pixel
value, onto the output. The stamps are placed $S$ pixels apart (the stride), and where they overlap
they are added. With stride 2, the output is about twice as large as the input. The filter is
learned, so the network learns *how* to upsample, unlike nearest-neighbour upsampling, which has no
parameters.

| | Output size (height or width) |
|---|---|
| `nn.Conv2d(…, kernel_size=F, stride=S, padding=P)` | $\left\lfloor \dfrac{N + 2P - F}{S} \right\rfloor + 1$ |
| `nn.ConvTranspose2d(…, kernel_size=F, stride=S, padding=P, output_padding=O)` | $(N - 1)\,S - 2P + F + O$ |

`output_padding` adds extra rows and columns on one side of the output. It is needed because a
strided convolution maps *several* input sizes to the same output size (both 13 and 14 become 7), so
the transposed convolution needs to be told which one you want.

### 🛠️ TASK 1a: Fix the decoder's shapes

✍️ Compute the output size of every layer by hand: the encoder goes 28 → ? → ?. Then set
`output_padding` (0 or 1) in the two transposed convolutions so that the decoder goes 7 → 14 → 28.

In [ ]:
class ConvAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7), nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=???), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=???),
            nn.Sigmoid())

    def encode(self, x):  return self.encoder(x)
    def decode(self, z):  return self.decoder(z)
    def forward(self, x): return self.decode(self.encode(x))

In [ ]:
# Test for Task 1a: print the shapes, layer by layer
_m = ConvAE(2).to(device)
_x = X_train[:4]
for layer in list(_m.encoder) + list(_m.decoder):
    _x = layer(_x)
    print(f'{layer.__class__.__name__:16s} -> {tuple(_x.shape)}')
assert _x.shape == (4, 1, 28, 28), 'the output must have the same shape as the input'
print('✅ Task 1a passed')

### Questions 1.1

1. How many learnable parameters does `nn.ConvTranspose2d(64, 32, 3)` have? Compare with the
   parameter-free alternative `nn.Upsample(scale_factor=2)`.
2. Where in the code is the **latent representation** produced, and what is its shape for a batch of
   $N$ images?

### Train a 2D autoencoder

With `latent_dim = 2`, every image is compressed to two numbers, so we can plot the latent space
directly, just as with the CNN embedding in 9A. The only difference: **no labels are used in
training**. We use the labels only to colour the plot afterwards.

In [ ]:
set_seed(0)
t0 = time.time()
ae2 = train_model(ConvAE(latent_dim=2).to(device), epochs=30)
print(f'Training time: {time.time() - t0:.0f} s')

with torch.no_grad():
    z_ae = ae2.encode(X_test)
plt.figure(figsize=(7, 6))
plot_latent(z_ae, y_test, title='Convolutional AE: 2D latent codes of the test images')
plt.show()

### 🛠️ TASK 1b: The latent map

The scatter plot shows where the encoder puts the images. The **latent map** shows the other
direction: we place a 20×20 grid of points in the latent space, **decode** each point, and show the
decoded image at its position. This tells us what the decoder produces *everywhere* in the latent
space, including the regions where there were no training images.

The grid must cover the region where the codes are. Set `xlim` and `ylim` to the 1st and 99th
percentiles of `z_ae[:, 0]` and `z_ae[:, 1]` (hint: `torch.quantile`).

In [ ]:
xlim = (???, ???)
ylim = (???, ???)

fig, axs = plt.subplots(1, 2, figsize=(16, 7))
plot_latent(z_ae, y_test, ax=axs[0], title='Where the encoder puts the test images', lim=(xlim, ylim))
plot_latent_grid(ae2.decode, xlim, ylim, ax=axs[1], title='What the decoder produces at each point')
plt.tight_layout(); plt.show()

### 🛠️ TASK 1c: Latent interpolation

Encode a 7 and a 9, walk along the straight line between their latent codes, and decode each point:
$\mathbf{z}(\alpha) = (1-\alpha)\,\mathbf{z}_7 + \alpha\,\mathbf{z}_9$ for $\alpha$ from 0 to 1. The
path is also drawn on the scatter plot, so you can see which regions it passes through.

In [ ]:
x7 = X_test[y_test == 7][:1]                         # one 7, shape (1, 1, 28, 28)
x9 = X_test[y_test == 9][:1]                         # one 9
alphas = torch.linspace(0, 1, 10, device=device).view(-1, 1)

with torch.no_grad():
    z7, z9 = ???, ???                                # encode both images: shape (1, 2) each
    z_path = ???                                     # the 10 interpolated latent vectors: shape (10, 2)
    x_path = ???                                     # decode them: shape (10, 1, 28, 28)

fig, axs = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [1, 1.6]})
plot_latent(z_ae, y_test, ax=axs[0], lim=(xlim, ylim), title='Interpolation path in the latent space')
axs[0].plot(*z_path.cpu().T, 'k.-', lw=2, ms=10)
show_images(x_path, nrow=10, ax=axs[1], title='Decoded images along the path: 7 → 9')
plt.tight_layout(); plt.show()

### Questions 1.2

1. Compare the scatter plot with the CNN embedding from 9A. How does a representation learned
   **without labels** differ from one learned with labels?
2. Look at the latent map. Which digits does the decoder produce well, and which are hard to find?
   What does it produce in the regions where the scatter plot has **no** points (the gaps, and the
   edges)?
3. Suppose you want to **generate** new digits by picking a random point in the latent space and
   decoding it. Which problems do you see? (Think about: which distribution would you sample from?
   What happens in the gaps?)
4. Does the interpolation path pass through other digit classes on its way from 7 to 9? Does every
   image along the path look like a plausible digit?

### More latent dimensions

Two dimensions are convenient for plotting, but very few for reconstruction. Let's train the same
model with `latent_dim = 16`. ✍️ **Prediction:** how much lower will the test loss be?

In [ ]:
set_seed(0)
ae16 = train_model(ConvAE(latent_dim=16).to(device), epochs=20)
print(f'test loss: latent_dim = 2: {evaluate(ae2):.1f}    latent_dim = 16: {evaluate(ae16):.1f}')
with torch.no_grad():
    show_images(torch.cat([X_test[:10], ae2(X_test[:10]), ae16(X_test[:10])]), nrow=10,
                title='input  ·  latent_dim = 2  ·  latent_dim = 16', figsize=(8, 2.8))

---
# Part 2 — A denoising autoencoder

A **denoising autoencoder** gets a corrupted image $\tilde{\mathbf{x}}$ as input, but its output is
compared with the **clean** image $\mathbf{x}$:

$$J = \frac{1}{n}\sum_i \left\| g\big(f(\tilde{\mathbf{x}}^{(i)})\big) - \mathbf{x}^{(i)} \right\|^2, \qquad \tilde{\mathbf{x}} = \mathbf{x} + \text{noise}.$$

To do this, the network cannot just copy its input: it must learn what digits look like, so it can
tell which parts of the input are digit and which are noise. (`train_model` above already does this
when `noise_std` is set: look for `x_in`.)

### 🛠️ TASK 2a: Corrupt the images

Add Gaussian noise with standard deviation `std`, and clip the result to the valid pixel range
$[0, 1]$ (hint: `torch.randn_like`, `.clamp`).

In [ ]:
def add_noise(x, std):
    return ???

In [ ]:
# Test for Task 2a
_x = X_test[:500]
_n = add_noise(_x, 0.4)
assert _n.shape == _x.shape and _n.min() >= 0 and _n.max() <= 1, 'the noisy images must stay in [0, 1]'
assert 0.15 < (_n - _x).abs().mean() < 0.35, 'the amount of noise looks wrong'
assert not torch.equal(add_noise(_x, 0.4), add_noise(_x, 0.4)), 'the noise should be random'
show_images(torch.cat([add_noise(X_test[:10], s) for s in [0.0, 0.2, 0.4, 0.6, 0.8]]), nrow=10,
            title='noise std 0.0 · 0.2 · 0.4 · 0.6 · 0.8', figsize=(8, 4.2))
print('✅ Task 2a passed')

### Train a denoising autoencoder

Same architecture as `ae16`, but trained on noisy inputs with `noise_std = 0.4`. Then we compare the
two models on test images with different amounts of noise.

✍️ **Prediction:** at noise level 0.4, which model has the lower test loss, and by how much? And on
**clean** test images (noise 0), will the denoising autoencoder be worse than `ae16`?

In [ ]:
set_seed(0)
dae = train_model(ConvAE(latent_dim=16).to(device), epochs=20, noise_std=0.4)

levels = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
print('noise std:              ' + ''.join(f'{s:8.1f}' for s in levels))
for name, m in [('AE (trained on clean)', ae16), ('denoising AE', dae)]:
    print(f'{name:24s}' + ''.join(f'{evaluate(m, s):8.1f}' for s in levels))

g = torch.Generator(device=device).manual_seed(1)
x_noisy = (X_test[:10] + 0.4 * torch.randn(X_test[:10].shape, generator=g, device=device)).clamp(0, 1)
with torch.no_grad():
    show_images(torch.cat([X_test[:10], x_noisy, ae16(x_noisy), dae(x_noisy)]), nrow=10,
                title='clean  ·  noisy input (0.4)  ·  AE (trained on clean)  ·  denoising AE', figsize=(8, 3.6))

### Questions 2

1. Compare the two rows of the table. How well does the normal autoencoder cope with noisy inputs,
   and how well does the denoising autoencoder?
2. What did the denoising autoencoder lose (if anything) on clean images? Why?
3. At which noise level does the denoising autoencoder break down? Look at the output images for a
   high noise level (change `0.4` to `0.8` in the last cell): when it fails, *how* does it fail?
4. Explain why the denoising autoencoder is forced to learn the structure of the data, while a normal
   autoencoder is not.
5. **Project perspective:** a denoising autoencoder fills in what a digit "should" look like. When
   would that be a problem, for example for medical or satellite images?

---
# Part 3 — Skip connections (U-Net)

All information in our autoencoders must pass through the bottleneck, so fine details are lost: the
outputs are smoother than the inputs. In Lecture 8 you saw how the **U-Net** solves this for
segmentation: **skip connections** pass feature maps from the encoder directly to the decoder layer
with the same resolution, where they are **concatenated** along the channel dimension.

Below is our convolutional autoencoder **plus two skip connections** (nothing else is changed):

```
input 1×28×28 ─ enc1 ─► 32×14×14 ─ enc2 ─► 64×7×7 ─► latent (16) ─► 64×7×7 ─┐
                           │                   └──────────── concat ─────────┴─► 128×7×7 ─ dec2 ─► 32×14×14 ─┐
                           └─────────────────────────────── concat ───────────────────────────────────────────┴─► 64×14×14 ─ dec1 ─► 1×28×28
```

### 🛠️ TASK 3a: Add the skip connections

Fill in the number of input channels of `dec2` and `dec1`, and the two concatenations
(`torch.cat([a, b], dim=1)` concatenates along the channel dimension). The argument `use_latent` lets
us switch the bottleneck off later.

In [ ]:
class SkipAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU())        # 32 x 14 x 14
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU())       # 64 x 7 x 7
        self.to_latent = nn.Sequential(nn.Flatten(), nn.Linear(64 * 7 * 7, latent_dim))
        self.from_latent = nn.Sequential(nn.Linear(latent_dim, 64 * 7 * 7), nn.ReLU(), nn.Unflatten(1, (64, 7, 7)))
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(???, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU())
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(???, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid())

    def forward(self, x, use_latent=True):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        z = self.to_latent(e2)
        if not use_latent:
            z = torch.zeros_like(z)                  # switch the bottleneck off
        d = self.from_latent(z)                      # 64 x 7 x 7
        d2 = self.dec2(???)                          # skip connection from enc2
        return self.dec1(???)                        # skip connection from enc1

In [ ]:
# Test for Task 3a
_m = SkipAE().to(device)
assert _m(X_train[:4]).shape == (4, 1, 28, 28)
print('Parameters: ConvAE(16):', sum(p.numel() for p in ConvAE(16).parameters()),
      '  SkipAE(16):', sum(p.numel() for p in _m.parameters()))
print('✅ Task 3a passed')

### Train it as a denoiser, and compare

We train the U-Net-style model exactly like the denoising autoencoder (same noise, epochs and seed).
Besides the loss, we measure **sharpness**: the mean absolute difference between neighbouring pixels
(large for crisp edges, small for blurry images), relative to the clean images. A value of 1.0 means
"as sharp as the real digits".

✍️ **Prediction:** which model has the lower test loss, and which produces sharper images?

In [ ]:
def sharpness(x):
    dx = (x[..., :, 1:] - x[..., :, :-1]).abs().mean()
    dy = (x[..., 1:, :] - x[..., :-1, :]).abs().mean()
    return (dx + dy).item()


set_seed(0)
skip = train_model(SkipAE(latent_dim=16).to(device), epochs=20, noise_std=0.4)

g = torch.Generator(device=device).manual_seed(1)
x_noisy = (X_test + 0.4 * torch.randn(X_test.shape, generator=g, device=device)).clamp(0, 1)
with torch.no_grad():
    for name, m in [('denoising AE', dae), ('denoising AE + skips', skip)]:
        print(f'{name:22s} test loss {evaluate(m, 0.4):6.1f}   sharpness {sharpness(m(x_noisy)) / sharpness(X_test):.2f}')
    show_images(torch.cat([X_test[:10], x_noisy[:10], dae(x_noisy[:10]), skip(x_noisy[:10])]), nrow=10,
                title='clean  ·  noisy (0.4)  ·  denoising AE  ·  denoising AE + skips', figsize=(8, 3.6))

### What happens if we switch the bottleneck off?

`use_latent=False` replaces the latent vector by zeros, so **no information passes through the
bottleneck**. ✍️ **Prediction:** what will each model output?

In [ ]:
with torch.no_grad():
    no_latent_ae = dae.decode(torch.zeros(10, 16, device=device))
    no_latent_skip = skip(x_noisy[:10], use_latent=False)
    loss_ae_off = recon_loss(dae.decode(torch.zeros(len(X_test), 16, device=device)), X_test).item()
    loss_skip_off = recon_loss(skip(x_noisy, use_latent=False), X_test).item()
print(f'denoising AE:          test loss with bottleneck {evaluate(dae, 0.4):5.1f},  without bottleneck {loss_ae_off:5.1f}')
print(f'denoising AE + skips:  test loss with bottleneck {evaluate(skip, 0.4):5.1f},  without bottleneck {loss_skip_off:5.1f}')
show_images(torch.cat([x_noisy[:10], no_latent_ae, no_latent_skip]), nrow=10,
            title='noisy input  ·  denoising AE, bottleneck off  ·  denoising AE + skips, bottleneck off', figsize=(8, 2.8))

### Questions 3

1. Did the skip connections improve the denoising? Use both measurements. Why do skip connections
   help with **fine details**?
2. What happened when the bottleneck was switched off, for each model? What does this tell you about
   where the information flows in the model with skip connections?
3. Suppose you trained the model with skip connections as a plain autoencoder (clean input → clean
   output) with a 2D latent space. Would you get a useful latent space for plotting or generating?
   Why not?
4. U-Nets are used for segmentation (Lecture 8) and inside diffusion models (Lecture 13), but not to
   learn a compact latent representation. Explain why, based on what you have seen.

---
# Part 4 — Exam-style questions

**Q1.** A `nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)` gets an
input of shape 64×7×7. What is the output shape?

* (a) 32×13×13 · (b) 32×14×14 · (c) 64×14×14 · (d) 32×4×4

**Q2.** What is the main difference between a transposed convolution and nearest-neighbour upsampling?

* (a) Only the transposed convolution can increase the spatial size.
* (b) The transposed convolution has learnable parameters; nearest-neighbour upsampling has none.
* (c) Nearest-neighbour upsampling can only be used in classifiers.
* (d) There is no difference; they are two names for the same operation.

**Q3.** A denoising autoencoder is trained with input $\tilde{\mathbf{x}} = \mathbf{x} + \text{noise}$.
What is its output compared with in the loss?

* (a) The noisy input $\tilde{\mathbf{x}}$.
* (b) The clean image $\mathbf{x}$.
* (c) The noise.
* (d) The class label of $\mathbf{x}$.

**Q4.** Why are decoded images from regions *between* the clusters of a standard autoencoder's latent
space often meaningless?

* (a) Because the decoder was never trained on latent vectors from those regions.
* (b) Because the latent space has too many dimensions.
* (c) Because the encoder uses ReLU activations.
* (d) Because the reconstruction loss was too low.

---
**Next: notebook 9C**, where we fix the latent space so that we can sample from it: the variational
autoencoder.